# 📘 Session 20: Saving, Loading, and Using ML Models
### Duration: ~2.5 Hours

---

**Topics Covered:**
1. End-to-End ML Workflow — Train, Evaluate, Persist, Reuse
2. Why Save Models? — Reproducibility and Handoff
3. Saving with `joblib` (Recommended for scikit-learn)
4. Saving with `pickle` — Standard Python Serialization
5. Loading Models and Making Predictions
6. Batch Inference on New Data
7. Model Metadata and Common Pitfalls

---

**Why This Session Matters for Data Science:**
- Training is only half the job — **deployed value** comes from **reusing** the fitted model
- **Pipelines must be saved with the model** so preprocessing is identical at inference time
- **`joblib`** and **`pickle`** are the two standard ways to persist Python/sklearn objects
- Teams hand off `.joblib` / `.pkl` files plus a short inference guide

> **Prerequisites:** Sessions 16–19 (pipelines, classification on penguins, regression on Boston housing).

> **Session 21 preview:** Hyperparameter tuning with `GridSearchCV` and validation curves.

**Libraries:** `pandas`, `numpy`, `seaborn`, `sklearn`, **`joblib`**, **`pickle`** (built-in)


---
## 1. End-to-End ML Workflow

```mermaid
flowchart LR
  eda[EDA] --> prep[Preprocess]
  prep --> train[TrainModel]
  train --> eval[Evaluate]
  eval --> save[SaveModel]
  save --> load[LoadLater]
  load --> infer[PredictNewData]
```

| Phase | Session | Output |
|-------|---------|--------|
| Understand data | 14–15 | EDA report |
| Prepare features | 16 | `Pipeline` preprocessor |
| Train & evaluate | 17–19 | Metrics, chosen model |
| **Persist & reuse** | **20 (today)** | `.joblib` / `.pkl` files |
| Tune hyperparameters | 21 | Best params via CV |

> **Data Science relevance**: A model file lets analysts, engineers, and notebooks share **one trained artifact** without re-running hours of training.


---
## 2. Why Save Models?

| Reason | Explanation |
|--------|-------------|
| **Speed** | Loading takes seconds; retraining can take minutes or hours |
| **Consistency** | Same weights, same preprocessing for every prediction |
| **Handoff** | Data scientist trains; app developer loads and calls `.predict()` |
| **Audit trail** | Pair saved model with metrics report from training date |

### What to save?

| Save this | Why |
|-----------|-----|
| **Full `Pipeline`** (preprocessor + model) | Recommended — new data gets same scaling/encoding |
| Model only | Only if preprocessing is duplicated elsewhere (risky) |

### Two serialization options (both common)

| Tool | Typical use |
|------|-------------|
| **`joblib`** | **Preferred for sklearn** — efficient for NumPy arrays |
| **`pickle`** | General Python objects — works for pipelines too; universal stdlib |

> ⚠️ **Special Case — security**: Never `pickle.load()` / `joblib.load()` files from **untrusted** sources (arbitrary code execution risk).


In [1]:
import pickle
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, r2_score, root_mean_squared_error

# Folder for saved artifacts (created relative to notebook working directory)
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
print("Model directory:", MODEL_DIR.resolve())


Model directory: C:\python_AI_ML\models


---
## 3. Train Two Example Models (Setup)

We fit **one classification pipeline** (penguins) and **one regression pipeline** (Boston) to use in save/load demos.


In [2]:
# --- Penguins classification pipeline (Sessions 16–17) ---
raw_pg = sns.load_dataset("penguins")
pg = raw_pg.dropna(
    subset=["species", "island", "sex", "bill_length_mm", "bill_depth_mm",
            "flipper_length_mm", "body_mass_g"]
).copy()
pg["bill_ratio"] = pg["bill_length_mm"] / pg["bill_depth_mm"]

y_pg = pg["species"]
X_pg = pg.drop(columns=["species"])
num_pg = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "bill_ratio"]
cat_pg = ["island", "sex"]

pre_pg = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_pg),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_pg),
])

clf_pipe = Pipeline([
    ("preprocessor", pre_pg),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42)),
])

X_pg_train, X_pg_test, y_pg_train, y_pg_test = train_test_split(
    X_pg, y_pg, test_size=0.2, random_state=42, stratify=y_pg,
)
clf_pipe.fit(X_pg_train, y_pg_train)
acc = accuracy_score(y_pg_test, clf_pipe.predict(X_pg_test))
print(f"Penguins classifier test accuracy: {acc:.3f}")


Penguins classifier test accuracy: 1.000


In [3]:
# --- Boston regression pipeline (Session 19) ---
from sklearn.datasets import fetch_openml

boston = fetch_openml(name="boston", version=1, as_frame=True, parser="auto")
X_bo = boston.data
y_bo = boston.target.astype(float)

X_bo_train, X_bo_test, y_bo_train, y_bo_test = train_test_split(
    X_bo, y_bo, test_size=0.2, random_state=42,
)

reg_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=200, random_state=42)),
])
reg_pipe.fit(X_bo_train, y_bo_train)
y_bo_pred = reg_pipe.predict(X_bo_test)
print(f"Boston regressor test R2: {r2_score(y_bo_test, y_bo_pred):.3f}")
print(f"Boston regressor test RMSE: {root_mean_squared_error(y_bo_test, y_bo_pred):.3f}")


Boston regressor test R2: 0.884
Boston regressor test RMSE: 2.918


---
## 4. Saving with `joblib` (Recommended)

scikit-learn documentation recommends **`joblib`** for persisting estimators and pipelines.

| Function | Action |
|----------|--------|
| `joblib.dump(obj, path)` | Write object to disk |
| `joblib.load(path)` | Read object back |

### Best practices

- Use extension **`.joblib`** for clarity
- Save the **entire fitted pipeline**
- Store **feature column list** separately if needed (DataFrame column order)


In [4]:
# Save classification and regression pipelines with joblib
pg_joblib_path = MODEL_DIR / "penguins_classifier.joblib"
bo_joblib_path = MODEL_DIR / "boston_regressor.joblib"

joblib.dump(clf_pipe, pg_joblib_path)
joblib.dump(reg_pipe, bo_joblib_path)

print("Saved:")
print(" ", pg_joblib_path, f"({pg_joblib_path.stat().st_size / 1024:.1f} KB)")
print(" ", bo_joblib_path, f"({bo_joblib_path.stat().st_size / 1024:.1f} KB)")


Saved:
  models\penguins_classifier.joblib (191.6 KB)
  models\boston_regressor.joblib (6848.3 KB)


---
## 5. Saving with `pickle` — Standard Library

**`pickle`** serializes almost any Python object. It is widely used in industry and works well for sklearn pipelines.

| Function | Action |
|----------|--------|
| `pickle.dump(obj, file)` | Write to open binary file |
| `pickle.load(file)` | Read from file |

### `joblib` vs `pickle` (quick comparison)

| Aspect | joblib | pickle |
|--------|--------|--------|
| **stdlib** | No (`pip install joblib`) | Yes (built-in) |
| **sklearn docs** | Recommended | Supported |
| **Large NumPy arrays** | Often faster/smaller | Works, can be slower |
| **Non-Python consumers** | No | Rarely |

> **Practical rule**: Use **joblib** for sklearn in new projects; know **pickle** because many legacy systems and tutorials use `.pkl` files.

> ⚠️ **Same security warning**: Only load pickle/joblib files you trust.


In [5]:
# Save the same pipelines with pickle (.pkl)
pg_pkl_path = MODEL_DIR / "penguins_classifier.pkl"
bo_pkl_path = MODEL_DIR / "boston_regressor.pkl"

with open(pg_pkl_path, "wb") as f:
    pickle.dump(clf_pipe, f)
with open(bo_pkl_path, "wb") as f:
    pickle.dump(reg_pipe, f)

print("Saved:")
print(" ", pg_pkl_path)
print(" ", bo_pkl_path)


Saved:
  models\penguins_classifier.pkl
  models\boston_regressor.pkl


---
## 6. Loading Models and Making Predictions

Simulate a **fresh session**: load from disk without refitting.

### Classification (penguins)

| Step | Code |
|------|------|
| Load | `model = joblib.load(path)` |
| Predict class | `model.predict(X_new)` |
| Probabilities | `model.predict_proba(X_new)` |

### Regression (Boston)

| Step | Code |
|------|------|
| Load | `model = joblib.load(path)` |
| Predict value | `model.predict(X_new)` |

New rows must have the **same feature columns** as training (names and types).

> 💡 **Tip — verifying reloads**: Classification `predict` returns **strings** (species names) — compare with `np.array_equal`. Regression returns **numbers** — use `np.allclose`.


In [7]:
# Load with joblib (as if in a new notebook / script)
clf_loaded = joblib.load(pg_joblib_path)
reg_loaded = joblib.load(bo_joblib_path)

# Verify same test predictions
# Classification: string labels — use array_equal (not allclose)
print("Reloaded clf matches?", np.array_equal(
    clf_loaded.predict(X_pg_test), clf_pipe.predict(X_pg_test)
))
# Regression: numeric predictions — allclose is fine
print("Reloaded reg matches?", np.allclose(
    reg_loaded.predict(X_bo_test), reg_pipe.predict(X_bo_test)
))

# Example: 2 new penguin rows (from test set for demo)
new_penguins = X_pg_test.head(2)
print("\nNew penguin predictions:", clf_loaded.predict(new_penguins))
print("Probabilities shape:", clf_loaded.predict_proba(new_penguins).shape)


Reloaded clf matches? True
Reloaded reg matches? True

New penguin predictions: ['Gentoo' 'Chinstrap']
Probabilities shape: (2, 3)


In [8]:
# Load with pickle — same objects, different file format
with open(pg_pkl_path, "rb") as f:
    clf_pkl = pickle.load(f)
with open(bo_pkl_path, "rb") as f:
    reg_pkl = pickle.load(f)

print("Pickle clf matches joblib?", np.array_equal(
    clf_pkl.predict(new_penguins), clf_loaded.predict(new_penguins)
))

new_boston = X_bo_test.head(3)
print("Boston predictions (pickle):", reg_pkl.predict(new_boston).round(2))


Pickle clf matches joblib? True
Boston predictions (pickle): [22.82 30.82 16.82]


---
## 7. Batch Inference and Metadata

### Batch predictions

Apply the model to many rows at once (CSV slice, database export, API batch).

### Optional metadata file

Save a small JSON/text note alongside the model: training date, metric, feature list, sklearn version.


In [9]:
import json
from datetime import datetime, timezone

# Batch: all test penguins
batch_pred = clf_loaded.predict(X_pg_test)
results_pg = X_pg_test.copy()
results_pg["true_species"] = y_pg_test.values
results_pg["predicted_species"] = batch_pred
results_pg["correct"] = results_pg["true_species"] == results_pg["predicted_species"]
print("Batch classification sample:\n", results_pg.head())

# Metadata sidecar for penguins model
meta = {
    "model_name": "penguins_species_classifier",
    "task": "classification",
    "target": "species",
    "feature_columns": list(X_pg.columns),
    "test_accuracy": float(acc),
    "saved_joblib": str(pg_joblib_path),
    "saved_pickle": str(pg_pkl_path),
    "sklearn_version": __import__("sklearn").__version__,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
meta_path = MODEL_DIR / "penguins_classifier_meta.json"
meta_path.write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("\nWrote metadata:", meta_path)


Batch classification sample:
      island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g  \
330  Biscoe            50.5           15.2              216.0       5000.0   
214   Dream            45.7           17.0              195.0       3650.0   
93    Dream            39.6           18.1              186.0       4450.0   
308  Biscoe            47.5           14.0              212.0       4875.0   
290  Biscoe            47.7           15.0              216.0       4750.0   

        sex  bill_ratio true_species predicted_species  correct  
330  Female    3.322368       Gentoo            Gentoo     True  
214  Female    2.688235    Chinstrap         Chinstrap     True  
93     Male    2.187845       Adelie            Adelie     True  
308  Female    3.392857       Gentoo            Gentoo     True  
290  Female    3.180000       Gentoo            Gentoo     True  

Wrote metadata: models\penguins_classifier_meta.json


---
## 🧪 Practice Exercises

1. Delete variables `clf_pipe` and `reg_pipe` from memory (or restart kernel). Load only from **joblib** and reproduce one test-set metric each.
2. Save **only** the inner `RandomForestClassifier` (not the full pipeline). Load it and try `predict` on raw `X_pg_test` — what goes wrong? Why must you save the pipeline?
3. Create a CSV with 3 fake Boston rows (use `X_bo_test.head(3)`), load the regressor with **pickle**, write predictions to a new column, save CSV.
4. Add `predict_proba` for the two demo penguin rows and show the probability for the predicted class.
5. Change one column name in `new_penguins` (e.g. rename a column) and call `predict` — observe the error and explain it.
6. Compare file sizes: `.joblib` vs `.pkl` for the same pipeline. Are they similar?
7. Write a 6-line **inference README** for a developer who receives `penguins_classifier.joblib` and a JSON metadata file.

---

## 📝 Session 20 Summary

| Topic | Key takeaway |
|-------|--------------|
| **Workflow** | Train → evaluate → **save** → load → predict |
| **joblib** | Preferred for sklearn pipelines (`dump` / `load`) |
| **pickle** | Built-in alternative; common `.pkl` files in industry |
| **Save pipeline** | Preprocessing + model together |
| **Inference** | Same columns as training; `predict` / `predict_proba` |
| **Metadata** | Document metrics, features, versions alongside files |

### Key gotchas
- Loading from **untrusted** pickle/joblib files is unsafe
- **sklearn version** differences can break loaded models after upgrades
- Saving **model only** without preprocessor causes wrong predictions
- **Column names and order** must match training data
- Test set is for **final evaluation** — do not tune using it, then save as "production" without documenting leakage

### Next Session
**Session 21**: Hyperparameter Tuning and Model Selection — `GridSearchCV`, `RandomizedSearchCV`, and validation curves for classification and regression.
